# 2. Base recommender

A causal transformer (GPT-2) with tied input and output item embeddings $E$ models the item sequence of a user:

$$\pi_\theta(i_{t+1} = j \mid i_{1:t}) = \operatorname{softmax}_j\,(E\, s_t), \qquad
\mathcal{L}_{\text{next}}(\theta) = -\sum_t \log \pi_\theta(i_{t+1} \mid i_{1:t}),$$

where $s_t$ is the state of the transformer after $t$ items. A list of $K$ items is produced in two ways:

- **top-K**: the $K$ most probable next items under $\pi_\theta(\cdot \mid h)$;
- **next-K**: item by item, $a_k = \arg\max_j \pi_\theta(j \mid h, a_{1:k-1})$, without repeating an item.

Relevance is measured on one request per user, issued at the start of the held-out period, against the user's
next $M$ events: all of them, those with $\rho \ge 0.5$ (*liked*) and those with $\rho < 0.5$ (*skipped*). The
metrics are Recall@K, NDCG@K, MAP@K and HitRate@K.

In [ ]:
import pandas as pd

from gptrec.data.batches import build_eval_examples
from gptrec.data.log import InteractionLog
from gptrec.evaluation.policy_report import EvaluationConfig, evaluate_policy, list_metrics, popularity_baseline_metrics
from gptrec.models.backbone import TransformerConfig
from gptrec.paths import ARTIFACTS_DIR, PROCESSED_DATA_DIR
from gptrec.rollout import GenerationConfig
from gptrec.training.checkpoint import save_model
from gptrec.training.precision import default_device
from gptrec.training.supervised import OptimizationConfig, PolicyTrainingConfig, train_policy

device = default_device()
log = InteractionLog.load(PROCESSED_DATA_DIR / "yambda-50m")

The model reads at most $L$ positions. A request spends $K$ of them on the generated items, which leaves $L - K$
for the history.

In [ ]:
MAX_POSITIONS = 128
HORIZON = 10
HISTORY_LEN = MAX_POSITIONS - HORIZON
MAX_FUTURE = 20

config = PolicyTrainingConfig(
    transformer=TransformerConfig(hidden_size=256, num_layers=4, num_heads=4, dropout=0.1, max_positions=MAX_POSITIONS),
    optimization=OptimizationConfig(epochs=10, batch_size=128, learning_rate=1e-3),
    window_stride=MAX_POSITIONS // 2,
)
evaluation_config = EvaluationConfig(ks=(1, 5, 10), generation=GenerationConfig(horizon=HORIZON, greedy=True))

validation_examples = build_eval_examples(log, "validation", HISTORY_LEN, MAX_FUTURE)
test_examples = build_eval_examples(log, "test", HISTORY_LEN, MAX_FUTURE)
print(f"{len(validation_examples):,} validation users, {len(test_examples):,} test users")

In [ ]:
def validation_relevance(policy) -> dict[str, float]:
    metrics = evaluate_policy(policy, validation_examples, log.items, {}, evaluation_config, device).metrics
    return {name: metrics[name] for name in ("top_k/all/ndcg@10", "next_k/all/ndcg@10")}


policy, history = train_policy(log, config, device, extra_validation=validation_relevance)
save_model(policy, ARTIFACTS_DIR / "policy_pretrained.pt")

In [ ]:
history = pd.DataFrame(history).set_index("epoch")
history[["train_loss", "validation_loss"]].plot(figsize=(6, 3), title="next-item cross-entropy")
history[["top_k/all/ndcg@10", "next_k/all/ndcg@10"]].plot(figsize=(6, 3), title="validation NDCG@10");

## Test period

The recommender against the most popular items. The difference between its two columns is the effect of
conditioning on generated items instead of observed ones.

In [ ]:
evaluation = evaluate_policy(policy, test_examples, log.items, {}, evaluation_config, device)
report = pd.DataFrame({
    "most popular": list_metrics(popularity_baseline_metrics(test_examples, log.items, evaluation_config), "top_k"),
    "GPT top-k": list_metrics(evaluation.metrics, "top_k"),
    "GPT next-k": list_metrics(evaluation.metrics, "next_k"),
})
with pd.option_context("display.max_rows", None, "display.float_format", "{:.4f}".format):
    display(report)